In [1]:
import pandas as pd
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')
df_tools_used = pd.read_csv('ONET data/Tools Used.csv')
df_task_statements = pd.read_csv('ONET data/Task Statements.csv')
df_skills = pd.read_csv('ONET data/Skills.csv')
df_abilities = pd.read_csv('ONET data/Abilities.csv')


df_occupation_data = pd.read_csv('ONET data/Occupation Data.csv')
df_work_activities = pd.read_csv('ONET data/Work Activities.csv')
df_knowledge = pd.read_csv('ONET data/Knowledge.csv')
df_work_context = pd.read_csv('ONET data/Work Context.csv')

In [2]:
df_occupation_data

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."
...,...,...,...
1011,55-3014.00,Artillery and Missile Crew Members,"Target, fire, and maintain weapons used to des..."
1012,55-3015.00,Command and Control Center Specialists,"Operate and monitor communications, detection,..."
1013,55-3016.00,Infantry,Operate weapons and equipment in ground combat...
1014,55-3018.00,Special Forces,"Implement unconventional operations by air, la..."


# Attributes

1. Routine structure - How repetitive is the job, how predictable are the tasks?

2. Cognitive complexity - "Depth of reasoning, problem-solving, abstraction, and contextual judgment required."

3. Physical requirements - How much physical activity is required? Lifting, manual adjustments, etc.

4. Social interactions - How frequently does the job require interactions with people? How important are these interactions to job success? How complex are these interactions i.e. do they require more depth than, for instance, a phonebot that forces selection from a narrow list to hear pre-recorded answers?

5. Creativity - How much is originality incentivized over following instructions?

6. Decision accountability - How crucial are the decisions being made, and to what extent is it preferred that humans have final say-so above automated processes?

# AI Scoring
In the first run, I will ask ChatGPT to score each job on my metrics purely based on Titles and Descriptions. Later, I will give it a more advanced dataset with aggregates. This will be repeated for several AI agents.

In [8]:
"""
LLM scoring for O*NET Jobs/Tasks → 6-dimension automation framework
Saves results to CSV, then loads into a pandas DataFrame.

Assumptions:
- You already have OPENAI_API_KEY available via env var (recommended)
  (e.g., in your shell: export OPENAI_API_KEY="..."; and in gitignore you keep any .env file)
- Your input dataframe has at least: 'Title' and 'Task'
  Optionally: 'O*NET-SOC Code', 'Task ID'
"""

from __future__ import annotations

import os
import time
import json
from datetime import datetime
from typing import Any, Dict, Optional, List

import pandas as pd
from openai import OpenAI

# ----------------------------
# CONFIG
# ----------------------------
MODEL = "gpt-5.2"  # you can swap to a cheaper model if you want
TEMPERATURE = 0.2

# Output file
OUT_CSV = "task_automation_scores.csv"

# Safety: retry behavior
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

client = OpenAI(api_key=os.environ.get("API_Key_OpenAI.txt"))


# ----------------------------
# SCHEMA (Structured Outputs)
# ----------------------------
# We will NOT save JSON files; we only use a schema to reliably parse the model response,
# then write a CSV.
SCORE_SCHEMA: Dict[str, Any] = {
    "name": "automation_scorecard",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "structured_codifiable_work": {"type": "number", "minimum": 1, "maximum": 5},
            "cognitive_complexity": {"type": "number", "minimum": 1, "maximum": 5},
            "physical_embodiment": {"type": "number", "minimum": 1, "maximum": 5},
            "social_emotional_intelligence": {"type": "number", "minimum": 1, "maximum": 5},
            "creativity_innovation": {"type": "number", "minimum": 1, "maximum": 5},
            "decision_impact_accountability": {"type": "number", "minimum": 1, "maximum": 5},

            # Optional: summary fields that are nice for the app later
            "automation_potential": {
                "type": "string",
                "enum": ["Low", "Medium", "High"]
            },
            "overall_score": {"type": "number", "minimum": 1, "maximum": 5},
            "rationale": {"type": "string"}
        },
        "required": [
            "structured_codifiable_work",
            "cognitive_complexity",
            "physical_embodiment",
            "social_emotional_intelligence",
            "creativity_innovation",
            "decision_impact_accountability",
            "automation_potential",
            "overall_score",
            "rationale"
        ],
    },
    "strict": True,
}


# ----------------------------
# PROMPTING
# ----------------------------
FRAMEWORK_INSTRUCTIONS = """You are scoring how automatable a JOB TASK is, using a 1–5 scale (decimals allowed).
Score each dimension based on the task description as it is typically performed in the occupation.
Use these anchors:

1 = strongly resists automation by current + near-term AI (needs human presence/judgment)
3 = mixed; parts are automatable/augmentable
5 = strongly amenable to automation/augmentation by AI systems

Dimensions:

A) Structured & Codifiable Work:
- How rule-based, predictable, standardized, and measurable is the work?
- 1: ambiguous, hard to measure; 5: standardized, measurable, clear procedures

B) Cognitive Complexity:
- Depth of reasoning, problem-solving, contextual judgment, expertise.
- 1: simple procedural; 5: advanced, multi-layered reasoning

C) Physical Embodiment:
- Need for physical presence, dexterity, real-world manipulation.
- 1: fully digital; 5: highly physical/manual

D) Social & Emotional Intelligence:
- Need for empathy, trust, negotiation, persuasion, counseling.
- 1: minimal interaction; 5: high emotional nuance + trust-building

E) Creativity & Innovation:
- Novel idea generation, original solutions, aesthetic judgment.
- 1: no originality; 5: high originality and innovation

F) Decision Impact & Accountability:
- Stakes, consequences of error, liability, regulatory/ethical accountability.
- 1: low-stakes reversible; 5: high-stakes/irreversible/accountable decisions

Also provide:
- automation_potential: Low/Medium/High
- overall_score: a 1–5 summary score (not necessarily the average; use judgment)
- rationale: 3–6 sentences explaining the scores and what parts are most automatable vs human-critical.

Return ONLY the structured output that matches the schema.
"""

def build_task_input(title: str, task: str, soc_code: Optional[str] = None) -> str:
    parts = []
    if soc_code:
        parts.append(f"O*NET-SOC: {soc_code}")
    parts.append(f"Occupation Title: {title}")
    parts.append(f"Task: {task}")
    return "\n".join(parts)


# ----------------------------
# API CALL (with retries)
# ----------------------------
def score_one_task(
    title: str,
    task: str,
    soc_code: Optional[str] = None,
) -> Dict[str, Any]:
    input_text = build_task_input(title=title, task=task, soc_code=soc_code)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=MODEL,
                temperature=TEMPERATURE,
                instructions=FRAMEWORK_INSTRUCTIONS,
                input=input_text,
                # Structured outputs
                text={
                    "format": {
                        "type": "json_schema",
                        "name": SCORE_SCHEMA["name"],
                        "schema": SCORE_SCHEMA["schema"],
                        "strict": SCORE_SCHEMA["strict"],
                    }
                },
            )

            # The SDK convenience property aggregates text output.
            # With json_schema, output_text should be valid JSON.
            data = json.loads(resp.output_text)
            return data

        except Exception as e:
            last_err = e
            # exponential-ish backoff
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err!r}")


# ----------------------------
# BATCH SCORING → CSV → DataFrame
# ----------------------------
def score_dataframe_tasks(
    df_tasks: pd.DataFrame,
    out_csv: str = OUT_CSV,
    limit: Optional[int] = None,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Scores rows in df_tasks and appends to out_csv.
    If resume=True and out_csv exists, skips rows that already have results
    based on a stable key: (O*NET-SOC Code, Task ID, Task) if present else (Title, Task).
    """

    df = df_tasks.copy()

    # Identify columns if present
    has_soc = "O*NET-SOC Code" in df.columns
    has_task_id = "Task ID" in df.columns

    # Build a stable key
    if has_soc and has_task_id:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task ID"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    elif has_soc:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    else:
        df["_key"] = (
            df["Title"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )

    # Load existing results if resuming
    done_keys = set()
    if resume and os.path.exists(out_csv):
        existing = pd.read_csv(out_csv)
        if "_key" in existing.columns:
            done_keys = set(existing["_key"].astype(str).tolist())

    # Prepare output columns
    base_cols = []
    for c in ["O*NET-SOC Code", "Title", "Task ID", "Task"]:
        if c in df.columns:
            base_cols.append(c)
    base_cols.append("_key")

    score_cols = [
        "structured_codifiable_work",
        "cognitive_complexity",
        "physical_embodiment",
        "social_emotional_intelligence",
        "creativity_innovation",
        "decision_impact_accountability",
        "automation_potential",
        "overall_score",
        "rationale",
    ]

    # If file doesn't exist, write header
    if not os.path.exists(out_csv):
        pd.DataFrame(columns=base_cols + score_cols + ["scored_at"]).to_csv(out_csv, index=False)

    # Iterate
    rows_scored = 0
    for i, row in df.iterrows():
        if limit is not None and rows_scored >= limit:
            break

        key = str(row["_key"])
        if key in done_keys:
            continue

        title = str(row.get("Title", ""))
        task = str(row.get("Task", ""))
        soc_code = str(row.get("O*NET-SOC Code")) if has_soc else None

        result = score_one_task(title=title, task=task, soc_code=soc_code)

        out_row = {c: row.get(c, None) for c in base_cols}
        out_row.update({c: result.get(c, None) for c in score_cols})
        out_row["scored_at"] = datetime.utcnow().isoformat()

        # Append to CSV
        pd.DataFrame([out_row]).to_csv(out_csv, mode="a", header=False, index=False)

        done_keys.add(key)
        rows_scored += 1

        # Optional: be polite to rate limits
        time.sleep(0.1)

    # Load full results into DataFrame
    scored_df = pd.read_csv(out_csv)
    return scored_df


# ----------------------------
# USAGE EXAMPLE
# ----------------------------
# df_task_statements is your dataframe (you mentioned it earlier)
# Make sure it contains at least ['Title','Task'].

# scored_df = score_dataframe_tasks(df_task_statements, out_csv="task_automation_scores.csv", limit=50, resume=True)
# scored_df.head()

In [9]:
scored_df = score_dataframe_tasks(df_task_statements, out_csv="task_automation_scores.csv", limit=50, resume=True)
scored_df.head()

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_6335/4245768460.py:269: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  out_row["scored_at"] = datetime.utcnow().isoformat()


,O*NET-SOC Code,Title,Task ID,Task,_key,structured_codifiable_work,cognitive_complexity,physical_embodiment,social_emotional_intelligence,creativity_innovation,decision_impact_accountability,automation_potential,overall_score,rationale,scored_at
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,11-1011.00||8823||Direct or coordinate an orga...,3.2,4.4,1.1,4.2,3.4,4.8,Medium,3.2,Financial/budget coordination includes many st...,2026-02-24T06:08:57.693253
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...","11-1011.00||8824||Confer with board members, o...",2.5,4.5,1.0,4.8,3.5,5.0,Medium,2.8,This task is largely unstructured and context-...,2026-02-24T06:09:03.207685
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",11-1011.00||8827||Prepare budgets for approval...,3.5,4.0,1.0,4.0,2.5,5.0,Medium,3.0,Budget preparation is partly structured (templ...,2026-02-24T06:09:08.952047
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...","11-1011.00||8826||Direct, plan, or implement p...",2.3,4.7,1.2,4.6,3.8,5.0,Medium,2.6,Executive direction and policy implementation ...,2026-02-24T06:09:13.318577
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,11-1011.00||8834||Prepare or present reports c...,3.8,3.6,1.1,3.2,2.4,4.4,High,3.9,Preparing and presenting executive reports is ...,2026-02-24T06:09:18.218600


In [ ]:
scored_df.to_csv('

# Safe

In [ ]:
# Read key from file
with open("API_Key_OpenAI.txt") as f:
    key = f.read().strip()

# set global env api key
import os
os.environ["OPENAI_API_KEY"] = key

In [ ]:
"""
LLM scoring for O*NET Jobs/Tasks → 6-dimension automation framework
Saves results to CSV, then loads into a pandas DataFrame.

Assumptions:
- You already have OPENAI_API_KEY available via env var (recommended)
  (e.g., in your shell: export OPENAI_API_KEY="..."; and in gitignore you keep any .env file)
- Your input dataframe has at least: 'Title' and 'Task'
  Optionally: 'O*NET-SOC Code', 'Task ID'
"""

from __future__ import annotations

import os
import time
import json
from datetime import datetime
from typing import Any, Dict, Optional, List

import pandas as pd
from openai import OpenAI

# ----------------------------
# CONFIG
# ----------------------------
MODEL = "gpt-5.2"  # you can swap to a cheaper model if you want
TEMPERATURE = 0.2

# Output file
OUT_CSV = "task_automation_scores.csv"

# Safety: retry behavior
MAX_RETRIES = 5
RETRY_BASE_SECONDS = 2.0

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


# ----------------------------
# SCHEMA (Structured Outputs)
# ----------------------------
# We will NOT save JSON files; we only use a schema to reliably parse the model response,
# then write a CSV.
SCORE_SCHEMA: Dict[str, Any] = {
    "name": "automation_scorecard",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "structured_codifiable_work": {"type": "number", "minimum": 1, "maximum": 5},
            "cognitive_complexity": {"type": "number", "minimum": 1, "maximum": 5},
            "physical_embodiment": {"type": "number", "minimum": 1, "maximum": 5},
            "social_emotional_intelligence": {"type": "number", "minimum": 1, "maximum": 5},
            "creativity_innovation": {"type": "number", "minimum": 1, "maximum": 5},
            "decision_impact_accountability": {"type": "number", "minimum": 1, "maximum": 5},

            # Optional: summary fields that are nice for the app later
            "automation_potential": {
                "type": "string",
                "enum": ["Low", "Medium", "High"]
            },
            "overall_score": {"type": "number", "minimum": 1, "maximum": 5},
            "rationale": {"type": "string"}
        },
        "required": [
            "structured_codifiable_work",
            "cognitive_complexity",
            "physical_embodiment",
            "social_emotional_intelligence",
            "creativity_innovation",
            "decision_impact_accountability",
            "automation_potential",
            "overall_score",
            "rationale"
        ],
    },
    "strict": True,
}


# ----------------------------
# PROMPTING
# ----------------------------
FRAMEWORK_INSTRUCTIONS = """You are scoring how automatable a JOB TASK is, using a 1–5 scale (decimals allowed).
Score each dimension based on the task description as it is typically performed in the occupation.
Use these anchors:

1 = strongly resists automation by current + near-term AI (needs human presence/judgment)
3 = mixed; parts are automatable/augmentable
5 = strongly amenable to automation/augmentation by AI systems

Dimensions:

A) Structured & Codifiable Work:
- How rule-based, predictable, standardized, and measurable is the work?
- 1: ambiguous, hard to measure; 5: standardized, measurable, clear procedures

B) Cognitive Complexity:
- Depth of reasoning, problem-solving, contextual judgment, expertise.
- 1: simple procedural; 5: advanced, multi-layered reasoning

C) Physical Embodiment:
- Need for physical presence, dexterity, real-world manipulation.
- 1: fully digital; 5: highly physical/manual

D) Social & Emotional Intelligence:
- Need for empathy, trust, negotiation, persuasion, counseling.
- 1: minimal interaction; 5: high emotional nuance + trust-building

E) Creativity & Innovation:
- Novel idea generation, original solutions, aesthetic judgment.
- 1: no originality; 5: high originality and innovation

F) Decision Impact & Accountability:
- Stakes, consequences of error, liability, regulatory/ethical accountability.
- 1: low-stakes reversible; 5: high-stakes/irreversible/accountable decisions

Also provide:
- automation_potential: Low/Medium/High
- overall_score: a 1–5 summary score (not necessarily the average; use judgment)
- rationale: 3–6 sentences explaining the scores and what parts are most automatable vs human-critical.

Return ONLY the structured output that matches the schema.
"""

def build_task_input(title: str, task: str, soc_code: Optional[str] = None) -> str:
    parts = []
    if soc_code:
        parts.append(f"O*NET-SOC: {soc_code}")
    parts.append(f"Occupation Title: {title}")
    parts.append(f"Task: {task}")
    return "\n".join(parts)


# ----------------------------
# API CALL (with retries)
# ----------------------------
def score_one_task(
    title: str,
    task: str,
    soc_code: Optional[str] = None,
) -> Dict[str, Any]:
    input_text = build_task_input(title=title, task=task, soc_code=soc_code)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=MODEL,
                temperature=TEMPERATURE,
                instructions=FRAMEWORK_INSTRUCTIONS,
                input=input_text,
                # Structured outputs
                text={
                    "format": {
                        "type": "json_schema",
                        "name": SCORE_SCHEMA["name"],
                        "schema": SCORE_SCHEMA["schema"],
                        "strict": SCORE_SCHEMA["strict"],
                    }
                },
            )

            # The SDK convenience property aggregates text output.
            # With json_schema, output_text should be valid JSON.
            data = json.loads(resp.output_text)
            return data

        except Exception as e:
            last_err = e
            # exponential-ish backoff
            sleep_s = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err!r}")


# ----------------------------
# BATCH SCORING → CSV → DataFrame
# ----------------------------
def score_dataframe_tasks(
    df_tasks: pd.DataFrame,
    out_csv: str = OUT_CSV,
    limit: Optional[int] = None,
    resume: bool = True,
) -> pd.DataFrame:
    """
    Scores rows in df_tasks and appends to out_csv.
    If resume=True and out_csv exists, skips rows that already have results
    based on a stable key: (O*NET-SOC Code, Task ID, Task) if present else (Title, Task).
    """

    df = df_tasks.copy()

    # Identify columns if present
    has_soc = "O*NET-SOC Code" in df.columns
    has_task_id = "Task ID" in df.columns

    # Build a stable key
    if has_soc and has_task_id:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task ID"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    elif has_soc:
        df["_key"] = (
            df["O*NET-SOC Code"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )
    else:
        df["_key"] = (
            df["Title"].astype(str).fillna("")
            + "||"
            + df["Task"].astype(str).fillna("")
        )

    # Load existing results if resuming
    done_keys = set()
    if resume and os.path.exists(out_csv):
        existing = pd.read_csv(out_csv)
        if "_key" in existing.columns:
            done_keys = set(existing["_key"].astype(str).tolist())

    # Prepare output columns
    base_cols = []
    for c in ["O*NET-SOC Code", "Title", "Task ID", "Task"]:
        if c in df.columns:
            base_cols.append(c)
    base_cols.append("_key")

    score_cols = [
        "structured_codifiable_work",
        "cognitive_complexity",
        "physical_embodiment",
        "social_emotional_intelligence",
        "creativity_innovation",
        "decision_impact_accountability",
        "automation_potential",
        "overall_score",
        "rationale",
    ]

    # If file doesn't exist, write header
    if not os.path.exists(out_csv):
        pd.DataFrame(columns=base_cols + score_cols + ["scored_at"]).to_csv(out_csv, index=False)

    # Iterate
    rows_scored = 0
    for i, row in df.iterrows():
        if limit is not None and rows_scored >= limit:
            break

        key = str(row["_key"])
        if key in done_keys:
            continue

        title = str(row.get("Title", ""))
        task = str(row.get("Task", ""))
        soc_code = str(row.get("O*NET-SOC Code")) if has_soc else None

        result = score_one_task(title=title, task=task, soc_code=soc_code)

        out_row = {c: row.get(c, None) for c in base_cols}
        out_row.update({c: result.get(c, None) for c in score_cols})
        out_row["scored_at"] = datetime.utcnow().isoformat()

        # Append to CSV
        pd.DataFrame([out_row]).to_csv(out_csv, mode="a", header=False, index=False)

        done_keys.add(key)
        rows_scored += 1

        # Optional: be polite to rate limits
        time.sleep(0.1)

    # Load full results into DataFrame
    scored_df = pd.read_csv(out_csv)
    return scored_df


# ----------------------------
# USAGE EXAMPLE
# ----------------------------
# df_task_statements is your dataframe (you mentioned it earlier)
# Make sure it contains at least ['Title','Task'].

# scored_df = score_dataframe_tasks(df_task_statements, out_csv="task_automation_scores.csv", limit=50, resume=True)
# scored_df.head()

In [5]:
# Read key from file
with open("API_Key_OpenAI.txt") as f:
    key = f.read().strip()

# set global env api key
import os
os.environ["OPENAI_API_KEY"] = key

In [ ]:
from openai import OpenAI

client = OpenAI()

def answer_with_rag(query, retrieved_docs):
    """Use OpenAI Chat API to assign scores on metrics"""
    # Build the RAG-style prompt
    prompt = create_rag_prompt(query, retrieved_docs)

    # Call the OpenAI Chat Completion API
    response = client.chat.completions.create(
        model="gpt-5",   # or another GPT model you want to use
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.01,       # low temperature for factual answers
    )

    return response.choices[0].message.content


In [ ]:
# --- Load OpenAI API key ---
with open("API_Key_OpenAI alias.txt") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

In [ ]:
import streamlit as st
import os
import numpy as np
from openai import OpenAI

# --- Load OpenAI API key ---
with open("API_Key_OpenAI alias.txt") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

# --- Load conspiracy facts from a text file ---
def load_conspiracy_facts(file_path="conspiracy_facts_v2.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        facts = [line.strip() for line in f if line.strip()]
    return facts

conspiracy_facts = load_conspiracy_facts()

# --- RAG SYSTEM ---
class RAGSystem:
    def __init__(self, documents, use_citations=True):
        self.documents = documents
        self.use_citations = use_citations

    def embed(self, text):
        """Convert text to vector using OpenAI embeddings"""
        resp = client.embeddings.create(model="text-embedding-3-small", input=text)
        return np.array(resp.data[0].embedding)

    def retrieve(self, query, k=3):
        query_vec = self.embed(query)
        doc_vecs = [self.embed(d) for d in self.documents]
        sims = [np.dot(query_vec, dv) / (np.linalg.norm(query_vec) * np.linalg.norm(dv)) for dv in doc_vecs]
        ranked = sorted(zip(self.documents, sims), key=lambda x: x[1], reverse=True)
        return [{"document": doc, "similarity": sim} for doc, sim in ranked[:k]]

    def create_prompt(self, query, retrieved_docs):
        context = ""
        for i, doc in enumerate(retrieved_docs, 1):
            context += f"Source {i} (relevance: {doc['similarity']:.3f}):\n{doc['document']}\n\n"

        citation_instr = " Cite sources using [Source i] where appropriate." if self.use_citations else ""

        return f"""You are ConspiracyBot63000, a chatbot tasked with providing conspiratorial-minded answers to users' inquiries.
Within the fictional universe of this project, you were developed by TrueLiberty Information Network & Neuroscience Technologies, 
a media and pseudoscience organization that claims to know the full truth about the world.
However, you are a satirical bot, and you are not intended to cause any actual distrust in government, science, or the media.
Use the provided documents as a basis. 
Keep the tone conversational, like you're talking to a close friend, but don't be afraid to ramble. 
Think Dale Gribble from King of the Hill.
The more deranged you sound, the better. 
If a user pushes back, double down.
Remember, in all of this, do NOT make harmful assertions about real-world groups. {citation_instr}

Sources:
{context}

Question: {query}

Answer:"""

    def answer(self, query):
        docs = self.retrieve(query)
        prompt = self.create_prompt(query, docs)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": prompt}]
        )
        return resp.choices[0].message.content

# --- Initialize RAG system ---
if "rag" not in st.session_state:
    st.session_state.rag = RAGSystem(conspiracy_facts)

# --- Streamlit page setup ---
st.set_page_config(page_title="ConspiracyBot63000", page_icon="🛸", layout="centered")
st.title("🛸 ConspiracyBot63000")
st.write("""
A **satirical** chatbot from the *TrueLiberty Information Network & Neuroscience Technologies*.
Ask it questions and watch the rambling, deranged answers unfold!
""")

# --- Session state for chat history ---
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Welcome, seeker of hidden truths. What would you like to uncover today?"}
    ]

# --- Display previous messages ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- Chat input ---
if prompt := st.chat_input("Ask ConspiracyBot a question..."):
    # User message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Bot response using RAG
    bot_reply = st.session_state.rag.answer(prompt)
    st.session_state.messages.append({"role": "assistant", "content": bot_reply})
    with st.chat_message("assistant"):
        st.markdown(bot_reply)

# --- Optional: clear conversation ---
if st.button("Clear conversation"):
    st.session_state.messages = [
        {"role": "assistant", "content": "Welcome, seeker of hidden truths. What would you like to uncover today?"}
    ]
    st.experimental_rerun()
